In [ ]:
# http.server 모듈 이용한 웹 서버 구현
from http.server import HTTPServer, BaseHTTPRequestHandler

# URL 구성요소별 피싱하는 모듈(URL:프로토콜, 도메인, 파라미터(쿼리))
from urllib import parse

# 웹 서버 홈 페이지 제작(첫 화면)
html_index = '''
    <!doctype html>
    <html>
        <body>
            <h1>BMI(체질량지수) Calculator Welcome Page</h1>
            <h3><a href="/get">GET 방식으로 파라미터 전달</a></h3>
            <h3><a href="/post">POST 방식으로 파라미터 전달</a></h3>
        </body>
    </html>
'''

# GET 요청으로 파라미터를 전송하는 웹 페이지 제작
html_get = '''
    <!doctype html>
    <html>
        <body>
            <h1>GET 방식으로 파라미터(키, 몸무게) 전송하기</h1>
            <form method="GET" action="/get/bmi">
                <p><input type="text" name="height" placeholder="키(Cm)를 입력하세요"></p>
                <p><input type="text" name="weight" placeholder="몸무게(Kg)를 입력하세요"></p>
                <p><input type="submit" value="전송하기"></p>
            </form>
        </body>
    </html>
'''

# POST 요청으로 파라미터를 전송하는 웹 페이지 제작
html_post = '''
    <!doctype html>
    <html>
        <body>
            <h1>POST 방식으로 파라미터 전송하기</h1>
            <form method="POST" action="/post/bmi">
                <p><input type="text" name="height" placeholder="키(Cm)를 입력하세요"></p>
                <p><input type="text" name="weight" placeholder="몸무게(Kg)를 입력하세요"></p>
                <p><input type="submit" value="전송하기"></p>
            </form>
        </body>
    </html>
'''

# base 핸들러 상속받아 자체핸들러 구현
class myHandler(BaseHTTPRequestHandler):
    def do_GET(self): # 빈 로직을 상속받아 메소드 오버라이딩으로 자체 구현
        print("GET 요청들어옴")
        # 경로의 끝이 루트('/')인 경우(기본요청)
        if self.path.endswith('/'):
            self.send_response(200) # status line(HTTP/1.1 200 OK)
            self.send_header("Content-type","text/html; charset=utf-8") # body 타입지정
            self.end_headers() # 더이상 헤더 없음, blank line 추가
            # 응답메시지의 바디 전송
            self.wfile.write(html_index.encode('utf-8'))

        # 경로의 끝이 루트('/get')인 경우
        if self.path.endswith('/get'):
            self.send_response(200) # status line(HTTP/1.1 200 OK)
            self.send_header("Content-type","text/html; charset=utf-8") # body 타입지정
            self.end_headers() # 더이상 헤더 없음, blank line 추가
            # 응답메시지의 바디 전송
            self.wfile.write(html_get.encode('utf-8'))

        # 경로의 끝이 루트('/post')인 경우
        if self.path.endswith('/post'):
            self.send_response(200) # status line(HTTP/1.1 200 OK)
            self.send_header("Content-type","text/html; charset=utf-8") # body 타입지정
            self.end_headers() # 더이상 헤더 없음, blank line 추가
            # 응답메시지의 바디 전송
            self.wfile.write(html_post.encode('utf-8'))

        # 경로의 시작이 루트('/get/bmi')인 경우(경로의 끝에 쿼리스트링 추가되어 있음)
        if self.path.startswith('/get/bmi'):
            # URL(/get/id?key=value)에서 쿼리 스트링 피싱(분리)
            url = parse.urlparse(self.path) # URL을 구성요소별 분리
            q_string = url.query # 쿼리스트링의 폼 데이터 저장(key=value)
            # parse_qs() : 폼 데이터는 기본적인 문자열 --> 딕셔너리로 변환
            params = parse.parse_qs(q_string)
            print(f"파라미터 딕셔너리변환: {params}")
            height = params.get('height',[''])[0] # height의 첫번째 값 추출
            weight = params.get('weight',[''])[0] # weight의 첫번째 값 추출
            print(f"height = {height}, weight = {weight}")

            # 추출한 name값을 포함해서 응답 페이지 보내기
            self.send_response(200) # status line(HTTP/1.1 200 OK)
            self.send_header("Content-type","text/html; charset=utf-8") # body 타입지정
            self.end_headers() # 더이상 헤더 없음, blank line 추가
            # 응답메시지의 바디 전송
            bmi = round(int(weight)/(int(height)/100)**2,2) # 소수점 2자리까지만 출력
            response_body = f"<h1>당신의 체질량지수는 {bmi} 입니다.</h1>"
            self.wfile.write(response_body.encode('utf-8'))

    def do_POST(self):
        print("POST 요청들어옴")
        # 경로의 끝이 루트('/post/id')인 경우
        if self.path.endswith('/post/bmi'):
            # 웹브라우저 요청메시지 바디의 폼데이터(파라미터) 저장
            content_length = int(self.headers['Content-length']) # 바디 길이저장
            data = self.rfile.read(content_length).decode('utf-8') # 바디 폼데이터 저장
            if data is not None:
                dic_data = parse.parse_qs(data)
                print(f"폼데이터 딕셔너리변환: {dic_data}")
                height = dic_data.get('height',[''])[0] # height의 첫번째 값 추출
                weight = dic_data.get('weight',[''])[0] # weight의 첫번째 값 추출
                
            # 바디에서 추출한 name값을 포함해서 응답 페이지 보내기
            self.send_response(200) # status line(HTTP/1.1 200 OK)
            self.send_header("Content-type","text/html; charset=utf-8") # body 타입지정
            self.end_headers() # 더이상 헤더 없음, blank line 추가
            # 응답메시지의 바디 전송
            bmi = round(int(weight)/(int(height)/100)**2,2) # 소수점 2자리까지만 출력
            response_body = f"<h1>당신의 체질량지수는 {bmi} 입니다.</h1>"
            self.wfile.write(response_body.encode('utf-8'))

if __name__ == "__main__":
    address = ('localhost',8080)
    # HTTP 서버 생성: HTTPServer(주소, 핸들러)
    server = HTTPServer(address,myHandler)
    print(f"웹 서버가 {address}로 서비스 되고 있습니다")
    server.serve_forever() # 서버시작(looping)

웹 서버가 ('localhost', 8080)로 서비스 되고 있습니다
GET 요청들어옴


127.0.0.1 - - [11/May/2026 22:05:31] "GET / HTTP/1.1" 200 -


GET 요청들어옴
GET 요청들어옴


127.0.0.1 - - [11/May/2026 22:05:33] "GET /post HTTP/1.1" 200 -


GET 요청들어옴
POST 요청들어옴
폼데이터 딕셔너리변환: {'height': ['170'], 'weight': ['60']}
GET 요청들어옴


...

GET 요청들어옴
GET 요청들어옴


127.0.0.1 - - [11/May/2026 22:05:58] "GET /get HTTP/1.1" 200 -


GET 요청들어옴
GET 요청들어옴
파라미터 딕셔너리변환: {'height': ['170'], 'weight': ['60']}
height = 170, weight = 60
GET 요청들어옴


127.0.0.1 - - [11/May/2026 22:06:02] "GET /get/bmi?height=170&weight=60 HTTP/1.1" 200 -


GET 요청들어옴
